1.Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns # draw confusion matrix
import matplotlib.pyplot as plt # draw graphs
import numpy as np # do mathematical computations
import kagglehub # download dataset
import pandas as pd # process file path table
import os
from PIL import Image # read images
import torch.nn.functional as F

2.Check GPU

In [2]:
# Before running this notebook, enabling GPU.
# Runtime->Change runtime type->Hardware accelerator: T4 GPU->Save
# This is to shorten training time since we have a large dataset
# Check if GPU is available for faster training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


3.Download Dataset

In [3]:
# download CIFAKE dataset from Kaggle
# "CIFAKE is a dataset that contains 60,000 synthetically-generated images and 60,000 real images (collected from CIFAR-10)."
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")

print("Path to dataset files:", path)
# verify dataset structure
print(os.listdir(path))
# check labels
print(os.listdir(path+'/train'))

Using Colab cache for faster access to the 'cifake-real-and-ai-generated-synthetic-images' dataset.
Path to dataset files: /kaggle/input/cifake-real-and-ai-generated-synthetic-images
['test', 'train']
['FAKE', 'REAL']


4.Build dataframe

In [4]:
# set up an empty list for collecting all image information
data=[]
# iterate through two folders 'test' and 'train'
for split in ['test','train']:
  # connect file path
  split_path=os.path.join(path,split)
  # check if the folder exists
  if os.path.exists(split_path):
    # iterate through 'FAKE' and 'REAL' subfolders
    for label in ['FAKE','REAL']:
      # connect file path
      label_path=os.path.join(split_path,label)
      # check if subfolders exist
      if os.path.exists(label_path):
        # scan all files in the folder
        with os.scandir(label_path) as entries:
          #iterate through all files
          for entry in entries:
            # make sure it only reads the image file
            if entry.is_file():
              data.append({
                  'file_path':entry.path, # file path
                  'label':1 if label=='FAKE' else 0, # FAKE=1 or REAL=0
                  'split':split # train or test
              })
# convert the file list into a table
df=pd.DataFrame(data)

print(f"Total images loaded: {len(df)}")
print(f"Train: {len(df[df['split']=='train'])}")
print(f"Test: {len(df[df['split']=='test'])}")
print(f"REAL: {len(df[df['label']==0])}")
print(f"FAKE: {len(df[df['label']==1])}")


Total images loaded: 120000
Train: 100000
Test: 20000
REAL: 60000
FAKE: 60000


5.Define a custom PyTorch Dataset

In [5]:
# referenced from PyTorch tutorial https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html
class CIFAKEDataset(Dataset):
  def __init__(self, dataframe, transform=None):
    self.dataframe = dataframe
    self.transform = transform

  def __len__(self):
    return len(self.dataframe) # the number of pictures in the dataset

  def __getitem__(self, idx):
    img_path = self.dataframe.iloc[idx, 0]
    image=Image.open(img_path).convert('RGB') # # open PIL Image (pic people see)
    label= self.dataframe.iloc[idx, 1] # 0=real, 1=fake
    if self.transform:
      image = self.transform(image) # transform PIL Image to tensor (numerical matrix)
    return image, label


6.Split into Train/Validation/Test sets

In [6]:
# Train: model learns from this data
# Validation: check performance during training to detect overfitting
# Test: final evaluation
# For this dataset, 80% of 'train' files are used for training; 20% of 'train' files are used for validation; all of 'test' files are used for testing

# shuffle train data and then seperate train and test
train=df[df['split']=='train'].sample(frac=1, random_state=42)
test=df[df['split']=='test']

# calculate split size
n_total=len(train)
n_train=int(0.8*n_total)
n_val=n_total-n_train

# slice into train and val
train_data=train[:n_train]
val_data=train[n_train:]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test)}")


Train: 80000, Val: 20000, Test: 20000


7.Define transform

In [7]:
transform=transforms.Compose([
    transforms.ToTensor(), # convert PIL images into Tensor， ToTensor() converts pixel value into range 0-1
    transforms.Normalize(mean=[0.5,0.5,0.5], # the average of R,G,B
                         # the normalization formula: (pixel-mean)/std
                         # normalizing it into range (-1)-1 makes the data better for model training
                         std=[0.5,0.5,0.5]) # the standard deviation of R,B,G
])

8.Create datasets and dataloaders

In [8]:
# referenced from L6_PyTorch_Model_Doodle_Classifer
BATCH_SIZE=64

train_dataset=CIFAKEDataset(train_data, transform=transform)
val_dataset=CIFAKEDataset(val_data, transform=transform)
test_dataset=CIFAKEDataset(test, transform=transform)

# feed data to the model batch by batch
train_loader=DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,num_workers=2) # shuffle=True prevents the model from remembering data sequence
val_loader=DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,num_workers=2)# num_workers helps prepare next batch, more time-efficient
test_loader=DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,num_workers=2)

# sanity check
images_batch, labels_batch = next(iter(train_loader))
print(f"One batch of images shape: {images_batch.shape}")
print(f"One batch of labels shape: {labels_batch.shape}")
print(f"Label examples: {labels_batch[:10]}")

One batch of images shape: torch.Size([64, 3, 32, 32])
One batch of labels shape: torch.Size([64])
Label examples: tensor([1, 0, 0, 1, 1, 0, 1, 0, 0, 0])


9.Build CNN model

In [9]:
# referenced from L6_PyTorch_Model_Doodle_Classifer
class CIFAKECNN(nn.Module):
  def __init__(self, labelnum=1):
    super(CIFAKECNN, self).__init__()

    # Block 1: 3 input channel->32 filters
    # formula: output_size=(input_size-kernel_size+2*padding)/stride+1
    # Input: (batch, 3, 32, 32)
    # After conv: (batch, 32, 30 ,30) <- (32-3+0)/1+1=30
    # After pool: (batch, 32, 15, 15) <- 30//2=15
    self.conv1=nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=0)
    self.pool=nn.MaxPool2d(kernel_size=2, stride=2)

    # Block 2: 32->64 filters
    # Input: (batch, 32, 15, 15)
    # After conv: (batch, 64, 13, 13) <- (15-3+0)/1+1=13
    # After pool: (batch, 64, 6, 6) <- 13//2=6
    self.conv2=nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=0)

    # Block 3: 64->128 filters
    # Input: (batch, 64, 6, 6)
    # After conv: (batch, 128, 4, 4) <- (6-3+0)/1+1=4
    # After pool: (batch, 128, 2, 2) <- 4//2=2
    self.conv3=nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=0)

    # 128*2*2 is the size after 3 convolutions and poolings, output is 256 features
    self.fc1=nn.Linear(128*2*2, 256)
    # turns off 50% nuerons to prevent overfitting
    self.dropout=nn.Dropout(0.5)
    # turns 256 features into 1 output
    self.fc2=nn.Linear(256, labelnum)

  def forward(self, x):
    # Block 1
    x=self.pool(F.relu(self.conv1(x))) # relu removes useless negative values
    # Block 2
    x=self.pool(F.relu(self.conv2(x)))
    # Block 3
    x=self.pool(F.relu(self.conv3(x)))
    # Flatten
    x=x.view(x.size(0),-1)
    # Fully connected
    x=F.relu(self.fc1(x))
    x=self.dropout(x)
    x=torch.sigmoid(self.fc2(x)) # compress value into 0-1
    return x

# instantiate model and move to GPU
model=CIFAKECNN().to(device)
print(model)

CIFAKECNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=512, out_features=256, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=256, out_features=1, bias=True)
)


10.Train the model

In [10]:
# loss function and optimizer
criterion=nn.BCELoss() # evaluate how wrong the model is
optimizer=optim.Adam(model.parameters(), lr=0.001) # adjust parameters

In [11]:
history = {'val_loss': [], 'val_acc': []} # Initialize history to store metrics

for epoch in range(10):
  model.train() # start training with dropout
  for batch_idx, (images, labels) in enumerate(train_loader): # take 64 images with label from train_loader every time
    images=images.to(device)
    labels=labels.to(device).float().unsqueeze(1) # Fix: Match output shape [batch_size, 1]
    # forward pass
    outputs=model(images) # feed images to the model and get predicted result (range from 0-1)
    # compute loss
    loss=criterion(outputs, labels)
    # backward pass
    optimizer.zero_grad() # clear the gradient of previous batch
    loss.backward() # calculate how each parameter should be adjusted
    # update weights
    optimizer.step()

  # validation
  model.eval() # turn off dropout
  with torch.no_grad():
    # initialize three counters
    val_loss=0
    correct=0
    total=0
    for images, labels in val_loader:
      images=images.to(device)
      labels=labels.to(device).float().unsqueeze(1) # Fix: Match output shape [batch_size, 1]
      outputs=model(images)
      loss=criterion(outputs, labels)
      val_loss+=loss.item() # cumulate loss
      # if >0.5-->Fake(1); otherwise Real(0)
      predicted=(outputs>0.5).float() # Predicted will also be [batch_size, 1]
      total+=labels.size(0) # Use labels.size(0) as it represents the batch size
      correct+=(predicted == labels).sum().item() # Fix: Correct comparison with labels of shape [batch_size, 1]

  average_loss=val_loss/total
  accuracy=100*correct/total

  history['val_loss'].append(average_loss) # Store metrics
  history['val_acc'].append(accuracy) # Store metrics

  print(f"Epoch {epoch+1}/10 | Val Loss: {average_loss:.4f} | Val Accuracy: {accuracy:.2f}%") # Fix: use average_loss
print(f"Best validation accuracy: {max(history['val_acc']):.1f}%")

Epoch 1/10 | Val Loss: 0.0045 | Val Accuracy: 87.94%
Epoch 2/10 | Val Loss: 0.0037 | Val Accuracy: 90.70%
Epoch 3/10 | Val Loss: 0.0029 | Val Accuracy: 92.67%
Epoch 4/10 | Val Loss: 0.0026 | Val Accuracy: 93.49%
Epoch 5/10 | Val Loss: 0.0030 | Val Accuracy: 92.93%
Epoch 6/10 | Val Loss: 0.0026 | Val Accuracy: 93.69%
Epoch 7/10 | Val Loss: 0.0025 | Val Accuracy: 94.06%
Epoch 8/10 | Val Loss: 0.0028 | Val Accuracy: 93.05%
Epoch 9/10 | Val Loss: 0.0029 | Val Accuracy: 93.46%
Epoch 10/10 | Val Loss: 0.0031 | Val Accuracy: 93.71%
Best validation accuracy: 94.1%
